# Forklift Anomaly Detection Skeleton

フォークリフト搭載ドラレコ動画から、まずは `edge persistence` を確認するためのノートブックです。

このノートブックでは、各章で関数を定義したあとにその場で処理を実行し、中間結果を確認できるようにしています。

0. パッケージのインストール
1. 動画の読み込み
1.5 暫定前処理（左右レターボックス除去）
2. 前後動画分割
2.5 ほぼ一色のフレーム除去（前後同期）
3. Edge Persistence の確認


## 0. パッケージのインストール

Dockerを使わずにノートブックを直接実行する場合は、最初にこのセルで依存パッケージを入れます。


In [ ]:
%pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 opencv-python-headless==4.11.0.86


In [ ]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display


In [ ]:
DATA_DIR = Path("../data")
REFERENCE_VIDEO_PATH = DATA_DIR / "sample.mp4"

FRAME_SAMPLING_FPS = 2
MAX_SAMPLE_FRAMES = 200
EDGE_PERSISTENCE_THRESHOLD = 0.50
EDGE_ENDPOINT_MAX_GAP = 100
EDGE_BARRIER_KERNEL_SIZE = 5
CONCAVE_HULL_ALPHA_RADIUS = 15.0
CONCAVE_HULL_POINT_SAMPLE_STEP = 3
MIN_EDGE_COMPONENT_PIXELS = 20
MIN_CONCAVE_HULL_REGION_AREA_RATIO = 0.005


## 1. 動画の読み込み

まずは参照動画を 1 本読み込み、前処理と `edge persistence` の確認に使う sampled frames を用意します。


In [ ]:
def read_video_metadata(video_path: Path) -> dict[str, Any]:
    # 動画全体の長さや解像度を先に取得して、後続セルの処理条件に使う。
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "video_path": str(video_path),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    metadata["duration_sec"] = (
        metadata["frame_count"] / metadata["fps"] if metadata["fps"] > 0 else np.nan
    )
    cap.release()
    return metadata


def iter_sampled_frames(
    video_path: Path,
    sampling_fps: int = FRAME_SAMPLING_FPS,
    max_frames: int | None = MAX_SAMPLE_FRAMES,
) -> list[np.ndarray]:
    # クリップ全体を均等に見るため、元 fps から間引いて sampled frames を作る。
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 1.0
    step = max(int(round(native_fps / max(sampling_fps, 1))), 1)

    frames: list[np.ndarray] = []
    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0
    while cap.isOpened():
        success, frame_bgr = cap.read()
        if not success:
            break

        if frame_idx % step == 0:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames) >= max_frames:
                break

        frame_idx += 1

    cap.release()
    return frames


def show_image(image: np.ndarray, title: str, figsize: tuple[int, int] = (8, 4)) -> None:
    # notebook 上で確実に画像表示できるよう、PNG にして埋め込む。
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
    buffer = BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


In [ ]:
reference_video_path = REFERENCE_VIDEO_PATH

print(f"reference video exists: {reference_video_path.exists()}")

reference_metadata = None
reference_sampled_stacked_frames = []
reference_stacked_frame = None

if not reference_video_path.exists():
    print(f"No reference video found at {reference_video_path}")
else:
    print(f"reference video: {reference_video_path}")
    reference_metadata = read_video_metadata(reference_video_path)
    print("video metadata:")
    print(reference_metadata)

    reference_sampled_stacked_frames = iter_sampled_frames(
        reference_video_path,
        sampling_fps=FRAME_SAMPLING_FPS,
    )
    print(f"sampled stacked frames: {len(reference_sampled_stacked_frames)}")

    if not reference_sampled_stacked_frames:
        print("No sampled frames could be loaded from the reference video.")
    else:
        reference_stacked_frame = reference_sampled_stacked_frames[0]
        print(f"stacked frame shape: {reference_stacked_frame.shape}")
        show_image(reference_stacked_frame, title="Loaded First Frame")


## 1.5 暫定: 左右レターボックス除去

この処理は、現時点の動画に左右の黒いレターボックスが含まれる前提で入れている**一時的な前処理**です。
入力動画の仕様が変わった場合は、このブロックを見直すか削除してください。


In [ ]:
def detect_side_letterbox_bounds(
    stacked_frame: np.ndarray,
    black_threshold: int = 8,
    min_non_black_ratio: float = 0.01,
) -> tuple[int, int, dict[str, int]]:
    # 各列に非黒画素がどれだけあるかを見て、左右の黒帯候補を求める。
    gray = cv2.cvtColor(stacked_frame, cv2.COLOR_RGB2GRAY)
    non_black_ratio = (gray > black_threshold).mean(axis=0)
    valid_columns = np.where(non_black_ratio > min_non_black_ratio)[0]

    if valid_columns.size == 0:
        crop_left = 0
        crop_right = stacked_frame.shape[1]
    else:
        crop_left = int(valid_columns[0])
        crop_right = int(valid_columns[-1]) + 1

    if crop_right <= crop_left:
        crop_left = 0
        crop_right = stacked_frame.shape[1]

    crop_info = {
        "original_width": int(stacked_frame.shape[1]),
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(stacked_frame.shape[1] - crop_right),
    }
    return crop_left, crop_right, crop_info


def apply_side_crop(stacked_frame: np.ndarray, crop_left: int, crop_right: int) -> np.ndarray:
    return stacked_frame[:, crop_left:crop_right, :]


def choose_common_side_letterbox_crop(
    stacked_frames: list[np.ndarray],
    left_percentile: float = 90.0,
    right_percentile: float = 10.0,
) -> dict[str, int]:
    # フレームごとに幅がずれないよう、クリップ全体で共通のクロップ幅を決める。
    if not stacked_frames:
        return {
            "frame_count": 0,
            "original_width": 0,
            "crop_left": 0,
            "crop_right": 0,
            "cropped_width": 0,
            "removed_left_px": 0,
            "removed_right_px": 0,
            "detected_crop_left_min": 0,
            "detected_crop_left_max": 0,
            "detected_crop_right_min": 0,
            "detected_crop_right_max": 0,
        }

    detected_crop_lefts: list[int] = []
    detected_crop_rights: list[int] = []
    original_width = int(stacked_frames[0].shape[1])

    for stacked_frame in stacked_frames:
        crop_left, crop_right, _ = detect_side_letterbox_bounds(stacked_frame)
        detected_crop_lefts.append(crop_left)
        detected_crop_rights.append(crop_right)

    # 少し保守的な percentile を使い、黒帯を残しにくくしつつ過剰クロップを避ける。
    crop_left = int(np.percentile(detected_crop_lefts, left_percentile))
    crop_right = int(np.percentile(detected_crop_rights, right_percentile))

    crop_left = max(0, min(crop_left, original_width - 1))
    crop_right = max(crop_left + 1, min(crop_right, original_width))

    # 極端に狭い結果は誤検出とみなし、中央値ベースに戻す。
    minimum_width = max(32, int(original_width * 0.2))
    if crop_right - crop_left < minimum_width:
        crop_left = int(np.median(detected_crop_lefts))
        crop_right = int(np.median(detected_crop_rights))
        crop_left = max(0, min(crop_left, original_width - 1))
        crop_right = max(crop_left + 1, min(crop_right, original_width))

    return {
        "frame_count": int(len(stacked_frames)),
        "original_width": original_width,
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(original_width - crop_right),
        "detected_crop_left_min": int(min(detected_crop_lefts)),
        "detected_crop_left_max": int(max(detected_crop_lefts)),
        "detected_crop_right_min": int(min(detected_crop_rights)),
        "detected_crop_right_max": int(max(detected_crop_rights)),
    }


def preprocess_stacked_frames_remove_letterbox(
    stacked_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], dict[str, int]]:
    # 1本の動画に対して同じ左右クロップを適用し、後段の frame stack を壊さないようにする。
    if not stacked_frames:
        return [], choose_common_side_letterbox_crop(stacked_frames)

    common_crop_info = choose_common_side_letterbox_crop(stacked_frames)
    crop_left = common_crop_info["crop_left"]
    crop_right = common_crop_info["crop_right"]
    cropped_frames = [apply_side_crop(stacked_frame, crop_left, crop_right) for stacked_frame in stacked_frames]
    return cropped_frames, common_crop_info


reference_preprocessed_stacked_frames = []
reference_preprocessed_frame = None
reference_letterbox_crop_info = None

if not reference_sampled_stacked_frames:
    print("Skip temporary letterbox removal because no sampled frames are loaded.")
else:
    reference_preprocessed_stacked_frames, reference_letterbox_crop_info = preprocess_stacked_frames_remove_letterbox(
        reference_sampled_stacked_frames
    )
    reference_preprocessed_frame = reference_preprocessed_stacked_frames[0]

    print("temporary common letterbox crop info:")
    print(reference_letterbox_crop_info)
    print(f"preprocessed stacked frames: {len(reference_preprocessed_stacked_frames)}")
    show_image(reference_stacked_frame, title="Before Temporary Letterbox Removal")
    show_image(reference_preprocessed_frame, title="After Temporary Letterbox Removal")



## 2. 前後動画分割

前後カメラが縦結合されている前提で、暫定前処理後の sampled frames を前カメラ画像と後カメラ画像に分割します。


In [ ]:
def split_front_rear_frame(stacked_frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # 縦結合動画の上半分を front、下半分を rear として扱う。
    height = stacked_frame.shape[0]
    midpoint = height // 2
    front_frame = stacked_frame[:midpoint, :, :]
    rear_frame = stacked_frame[midpoint:, :, :]
    return front_frame, rear_frame


def split_front_rear_frames(stacked_frames: list[np.ndarray]) -> tuple[list[np.ndarray], list[np.ndarray]]:
    # sampled frames 全体を前後2系列へ分離する。
    front_frames: list[np.ndarray] = []
    rear_frames: list[np.ndarray] = []
    for stacked_frame in stacked_frames:
        front_frame, rear_frame = split_front_rear_frame(stacked_frame)
        front_frames.append(front_frame)
        rear_frames.append(rear_frame)
    return front_frames, rear_frames



In [ ]:
reference_front_frames = []
reference_rear_frames = []
reference_front_frame = None
reference_rear_frame = None

if not reference_preprocessed_stacked_frames:
    print("Skip split preview because no preprocessed frames are available.")
else:
    reference_front_frames, reference_rear_frames = split_front_rear_frames(reference_preprocessed_stacked_frames)
    print(f"split frame pairs: {len(reference_front_frames)}")

    reference_front_frame = reference_front_frames[0]
    reference_rear_frame = reference_rear_frames[0]
    print(f"front frame shape: {reference_front_frame.shape}")
    print(f"rear frame shape: {reference_rear_frame.shape}")
    show_image(reference_front_frame, title="Front First Frame Before Uniform-Frame Trim")
    show_image(reference_rear_frame, title="Rear First Frame Before Uniform-Frame Trim")


## 2.5 ほぼ一色のフレーム除去（前後同期）

動画の冒頭や末尾に、ほぼ一色で何も映っていないフレームが含まれることがあるため、前後分割後にこれを除去します。
前後動画のフレーム位置を合わせるため、先頭側・末尾側ともに front / rear のうち多く検出された枚数を採用し、両方から同じ枚数だけ削除します。


In [ ]:
def is_nearly_uniform_frame(
    frame: np.ndarray,
    dominant_ratio_threshold: float = 0.95,
    color_tolerance: float = 30.0,
) -> bool:
    pixels = frame.reshape(-1, 3).astype(np.float32)

    # Use the per-channel median as a robust representative color so a small number of outliers do not dominate the decision.
    representative_color = np.median(pixels, axis=0)

    # Count how many pixels stay close to that representative color across all RGB channels.
    pixel_deviation = np.abs(pixels - representative_color).max(axis=1)
    near_representative_ratio = float((pixel_deviation <= color_tolerance).mean())
    return near_representative_ratio >= dominant_ratio_threshold


def count_edge_uniform_frames(frames: list[np.ndarray], reverse: bool = False) -> int:
    # 先頭側または末尾側から、ほぼ一色のフレームが何枚続くかを数える。
    ordered_frames = list(reversed(frames)) if reverse else frames
    count = 0
    for frame in ordered_frames:
        if is_nearly_uniform_frame(frame):
            count += 1
        else:
            break
    return count


def trim_synchronized_uniform_frames(
    front_frames: list[np.ndarray],
    rear_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], list[np.ndarray], dict[str, int]]:
    # 前後で時刻を揃えるため、片側だけ空フレームでも両方から同数を落とす。
    if len(front_frames) != len(rear_frames):
        raise ValueError("Front and rear frame counts must match before uniform-frame trimming.")

    total_frames = len(front_frames)
    if total_frames == 0:
        trim_info = {
            "input_frame_pairs": 0,
            "front_uniform_frames_at_start": 0,
            "rear_uniform_frames_at_start": 0,
            "front_uniform_frames_at_end": 0,
            "rear_uniform_frames_at_end": 0,
            "removed_from_start": 0,
            "removed_from_end": 0,
            "remaining_frame_pairs": 0,
        }
        return [], [], trim_info

    front_uniform_start = count_edge_uniform_frames(front_frames, reverse=False)
    rear_uniform_start = count_edge_uniform_frames(rear_frames, reverse=False)
    front_uniform_end = count_edge_uniform_frames(front_frames, reverse=True)
    rear_uniform_end = count_edge_uniform_frames(rear_frames, reverse=True)

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)

    if removed_from_start + removed_from_end >= total_frames:
        trimmed_front_frames = []
        trimmed_rear_frames = []
    else:
        end_index = total_frames - removed_from_end if removed_from_end > 0 else total_frames
        trimmed_front_frames = front_frames[removed_from_start:end_index]
        trimmed_rear_frames = rear_frames[removed_from_start:end_index]

    trim_info = {
        "input_frame_pairs": int(total_frames),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(len(trimmed_front_frames)),
    }
    return trimmed_front_frames, trimmed_rear_frames, trim_info



In [ ]:
reference_uniform_trim_info = None

if not reference_front_frames or not reference_rear_frames:
    print("Skip synchronized uniform-frame trim because split frames are not available.")
else:
    reference_front_frames, reference_rear_frames, reference_uniform_trim_info = trim_synchronized_uniform_frames(
        reference_front_frames,
        reference_rear_frames,
    )
    print("uniform frame trim info:")
    print(reference_uniform_trim_info)

    if not reference_front_frames:
        reference_front_frame = None
        reference_rear_frame = None
        print("All frame pairs were removed by synchronized uniform-frame trimming.")
    else:
        reference_front_frame = reference_front_frames[0]
        reference_rear_frame = reference_rear_frames[0]
        print(f"aligned frame pairs after uniform-frame trim: {len(reference_front_frames)}")
        show_image(reference_front_frame, title="Front First Frame After Uniform-Frame Trim")
        show_image(reference_rear_frame, title="Rear First Frame After Uniform-Frame Trim")


## 3. Edge Persistence の確認

ここでは前後動画それぞれについて `edge persistence` を取得し、補間後エッジから凹包ベースの車体フレーム候補領域までをログ出力します。

- 各 sampled frame でエッジを検出する
- 時間方向に何枚のフレームでそのエッジが出たかを `edge persistence` として集計する
- その結果を画像として確認し、一定しきい値で二値化した mask も確認する
- 二値化後の mask の端点を補間し、離れたエッジを接続する
- 補間後エッジを barrier にし、連結成分ごとに凹包近似を計算する
- 一定以上の面積を持つ凹包領域だけを車体フレーム候補として採用する

これはまだ暫定のルールベース抽出で、後から視点別ルールや距離制約を追加できるようにしています。


In [ ]:
def sample_frames_evenly(frames: list[np.ndarray], max_samples: int = 12) -> tuple[list[np.ndarray], list[int]]:
    # clip 先頭だけに偏らないよう、全体から等間隔にフレームを抜き出す。
    if not frames:
        return [], []
    if len(frames) <= max_samples:
        return frames, list(range(len(frames)))

    sampled_indices = np.linspace(0, len(frames) - 1, num=max_samples, dtype=int)
    sampled_indices = np.unique(sampled_indices)
    return [frames[index] for index in sampled_indices], [int(index) for index in sampled_indices]


def normalize_map_to_rgb(image_2d: np.ndarray) -> np.ndarray:
    array = image_2d.astype(np.float32)
    if array.size == 0:
        return np.zeros((1, 1, 3), dtype=np.uint8)
    min_value = float(array.min())
    max_value = float(array.max())
    if max_value > min_value:
        array = (array - min_value) / (max_value - min_value)
    else:
        array = np.zeros_like(array)
    array_uint8 = (array * 255).astype(np.uint8)
    return cv2.cvtColor(array_uint8, cv2.COLOR_GRAY2RGB)


def draw_mask_overlay(
    image: np.ndarray,
    mask: np.ndarray | None,
    color: tuple[int, int, int] = (255, 0, 0),
    alpha: float = 0.35,
) -> np.ndarray:
    canvas = image.copy()
    if mask is None or not np.any(mask):
        return canvas
    color_array = np.array(color, dtype=np.float32)
    canvas_float = canvas.astype(np.float32)
    canvas_float[mask] = (1.0 - alpha) * canvas_float[mask] + alpha * color_array
    return np.clip(canvas_float, 0, 255).astype(np.uint8)


def find_edge_endpoints(mask: np.ndarray) -> np.ndarray:
    # 8近傍で隣接エッジが 1 個以下の画素を端点とみなす。
    if mask.size == 0:
        return mask.astype(bool)
    mask_float = mask.astype(np.float32)
    neighbor_count = cv2.filter2D(
        mask_float,
        cv2.CV_32F,
        np.ones((3, 3), dtype=np.float32),
        borderType=cv2.BORDER_CONSTANT,
    ) - mask_float
    endpoint_mask = (mask_float > 0) & (neighbor_count <= 1.0)
    return endpoint_mask.astype(bool)


def interpolate_edge_endpoints(mask: np.ndarray, max_gap: int = EDGE_ENDPOINT_MAX_GAP) -> tuple[np.ndarray, np.ndarray]:
    # 二値エッジの端点同士だけを 1 回ずつ結び、切れたエッジを補間する。
    if mask.size == 0:
        return mask.astype(bool), mask.astype(bool)
    endpoint_mask = find_edge_endpoints(mask)
    endpoint_coords = np.column_stack(np.where(endpoint_mask))
    if len(endpoint_coords) < 2:
        return mask.astype(bool), endpoint_mask.astype(bool)

    interpolated_mask = mask.astype(np.uint8).copy()
    used = np.zeros(len(endpoint_coords), dtype=bool)
    candidate_pairs: list[tuple[float, int, int]] = []
    for i in range(len(endpoint_coords)):
        y1, x1 = endpoint_coords[i]
        for j in range(i + 1, len(endpoint_coords)):
            y2, x2 = endpoint_coords[j]
            distance = float(np.hypot(float(y2 - y1), float(x2 - x1)))
            if 1.0 < distance <= float(max_gap):
                candidate_pairs.append((distance, i, j))

    candidate_pairs.sort(key=lambda item: item[0])
    for _, i, j in candidate_pairs:
        if used[i] or used[j]:
            continue
        y1, x1 = endpoint_coords[i]
        y2, x2 = endpoint_coords[j]
        cv2.line(interpolated_mask, (int(x1), int(y1)), (int(x2), int(y2)), color=1, thickness=1)
        used[i] = True
        used[j] = True

    return interpolated_mask > 0, endpoint_mask.astype(bool)


def build_edge_barrier_mask(mask: np.ndarray, kernel_size: int = EDGE_BARRIER_KERNEL_SIZE) -> np.ndarray:
    # 凹包近似の入力に使うため、補間後エッジを少し太らせる。
    if mask.size == 0:
        return mask.astype(bool)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    barrier_mask = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1)
    return barrier_mask > 0


def alpha_shape_mask_from_points(
    points_yx: np.ndarray,
    image_shape: tuple[int, int],
    alpha_radius: float = CONCAVE_HULL_ALPHA_RADIUS,
    point_sample_step: int = CONCAVE_HULL_POINT_SAMPLE_STEP,
) -> dict[str, Any]:
    # Delaunay 三角形の外接円半径で絞り込み、凹包近似の filled mask を作る。
    height, width = image_shape
    empty_mask = np.zeros((height, width), dtype=bool)
    if points_yx.size == 0:
        return {
            "concave_hull_mask": empty_mask,
            "inserted_point_count": 0,
            "kept_triangle_count": 0,
        }

    sampled_points_yx = points_yx[::max(1, int(point_sample_step))]
    if len(sampled_points_yx) < 3:
        return {
            "concave_hull_mask": empty_mask,
            "inserted_point_count": int(len(sampled_points_yx)),
            "kept_triangle_count": 0,
        }

    sampled_points_xy = np.unique(sampled_points_yx[:, ::-1], axis=0)
    if len(sampled_points_xy) < 3:
        return {
            "concave_hull_mask": empty_mask,
            "inserted_point_count": int(len(sampled_points_xy)),
            "kept_triangle_count": 0,
        }

    subdiv = cv2.Subdiv2D((0, 0, int(width), int(height)))
    inserted_point_count = 0
    for x_value, y_value in sampled_points_xy:
        x_clipped = int(np.clip(int(x_value), 0, width - 1))
        y_clipped = int(np.clip(int(y_value), 0, height - 1))
        try:
            subdiv.insert((float(x_clipped), float(y_clipped)))
            inserted_point_count += 1
        except cv2.error:
            continue

    if inserted_point_count < 3:
        return {
            "concave_hull_mask": empty_mask,
            "inserted_point_count": int(inserted_point_count),
            "kept_triangle_count": 0,
        }

    triangle_list = subdiv.getTriangleList()
    if triangle_list is None or len(triangle_list) == 0:
        return {
            "concave_hull_mask": empty_mask,
            "inserted_point_count": int(inserted_point_count),
            "kept_triangle_count": 0,
        }

    concave_hull_mask = np.zeros((height, width), dtype=np.uint8)
    kept_triangle_count = 0
    for triangle in triangle_list.reshape(-1, 6):
        points_xy = np.array(
            [
                [triangle[0], triangle[1]],
                [triangle[2], triangle[3]],
                [triangle[4], triangle[5]],
            ],
            dtype=np.float32,
        )
        if (
            np.any(points_xy[:, 0] < 0)
            or np.any(points_xy[:, 0] >= width)
            or np.any(points_xy[:, 1] < 0)
            or np.any(points_xy[:, 1] >= height)
        ):
            continue

        side_a = float(np.linalg.norm(points_xy[1] - points_xy[0]))
        side_b = float(np.linalg.norm(points_xy[2] - points_xy[1]))
        side_c = float(np.linalg.norm(points_xy[0] - points_xy[2]))
        triangle_area = float(abs(np.cross(points_xy[1] - points_xy[0], points_xy[2] - points_xy[0])) * 0.5)
        if triangle_area <= 1e-6:
            continue

        circumradius = (side_a * side_b * side_c) / (4.0 * triangle_area)
        if circumradius > float(alpha_radius):
            continue

        cv2.fillConvexPoly(concave_hull_mask, np.round(points_xy).astype(np.int32), color=1)
        kept_triangle_count += 1

    return {
        "concave_hull_mask": concave_hull_mask > 0,
        "inserted_point_count": int(inserted_point_count),
        "kept_triangle_count": int(kept_triangle_count),
    }


def build_concave_hull_candidate_mask(
    barrier_mask: np.ndarray,
    alpha_radius: float = CONCAVE_HULL_ALPHA_RADIUS,
    point_sample_step: int = CONCAVE_HULL_POINT_SAMPLE_STEP,
    min_edge_component_pixels: int = MIN_EDGE_COMPONENT_PIXELS,
) -> dict[str, Any]:
    # barrier の連結成分ごとに凹包近似を計算し、局所的な候補領域として合成する。
    if barrier_mask.size == 0:
        empty_mask = barrier_mask.astype(bool)
        return {
            "concave_hull_candidate_mask": empty_mask,
            "edge_component_count": 0,
            "used_edge_component_count": 0,
            "inserted_point_count": 0,
            "kept_triangle_count": 0,
        }

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        barrier_mask.astype(np.uint8),
        connectivity=8,
    )

    concave_hull_candidate_mask = np.zeros_like(barrier_mask, dtype=bool)
    used_edge_component_count = 0
    inserted_point_count = 0
    kept_triangle_count = 0

    for label in range(1, num_labels):
        component_area_pixels = int(stats[label, cv2.CC_STAT_AREA])
        if component_area_pixels < int(min_edge_component_pixels):
            continue

        component_points_yx = np.column_stack(np.where(labels == label))
        component_hull_info = alpha_shape_mask_from_points(
            component_points_yx,
            image_shape=barrier_mask.shape,
            alpha_radius=alpha_radius,
            point_sample_step=point_sample_step,
        )
        component_hull_mask = component_hull_info["concave_hull_mask"]
        if not np.any(component_hull_mask):
            continue

        concave_hull_candidate_mask |= component_hull_mask
        used_edge_component_count += 1
        inserted_point_count += int(component_hull_info["inserted_point_count"])
        kept_triangle_count += int(component_hull_info["kept_triangle_count"])

    return {
        "concave_hull_candidate_mask": concave_hull_candidate_mask.astype(bool),
        "edge_component_count": int(max(0, num_labels - 1)),
        "used_edge_component_count": int(used_edge_component_count),
        "inserted_point_count": int(inserted_point_count),
        "kept_triangle_count": int(kept_triangle_count),
    }


def select_large_concave_hull_regions(
    candidate_mask: np.ndarray,
    min_area_ratio: float = MIN_CONCAVE_HULL_REGION_AREA_RATIO,
) -> dict[str, Any]:
    # 凹包候補から面積の小さい領域を除き、車体フレーム候補だけを残す。
    if candidate_mask.size == 0:
        empty_mask = candidate_mask.astype(bool)
        return {
            "vehicle_frame_region_mask": empty_mask,
            "min_region_area_pixels": 0,
            "selected_region_count": 0,
        }

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        candidate_mask.astype(np.uint8),
        connectivity=8,
    )
    vehicle_frame_region_mask = np.zeros_like(candidate_mask, dtype=bool)
    min_region_area_pixels = max(1, int(np.ceil(float(min_area_ratio) * candidate_mask.size)))
    selected_region_count = 0

    for label in range(1, num_labels):
        area_pixels = int(stats[label, cv2.CC_STAT_AREA])
        if area_pixels < min_region_area_pixels:
            continue
        vehicle_frame_region_mask |= labels == label
        selected_region_count += 1

    return {
        "vehicle_frame_region_mask": vehicle_frame_region_mask.astype(bool),
        "min_region_area_pixels": int(min_region_area_pixels),
        "selected_region_count": int(selected_region_count),
    }


def fill_mask_holes(mask: np.ndarray) -> dict[str, Any]:
    # 候補領域の内側に閉じた背景成分だけを hole とみなし、面積しきい値の前に埋める。
    if mask.size == 0:
        empty_mask = mask.astype(bool)
        return {
            "filled_mask": empty_mask,
            "hole_mask": empty_mask,
            "filled_hole_count": 0,
            "filled_hole_area_pixels": 0,
        }

    background_mask = ~mask.astype(bool)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        background_mask.astype(np.uint8),
        connectivity=8,
    )

    border_labels: set[int] = set()
    if num_labels > 1:
        for border_values in (
            labels[0, :],
            labels[-1, :],
            labels[:, 0],
            labels[:, -1],
        ):
            for label in np.unique(border_values):
                label_int = int(label)
                if label_int != 0:
                    border_labels.add(label_int)

    hole_mask = np.zeros_like(mask, dtype=bool)
    filled_hole_count = 0
    filled_hole_area_pixels = 0
    for label in range(1, num_labels):
        if label in border_labels:
            continue
        region_mask = labels == label
        hole_mask |= region_mask
        filled_hole_count += 1
        filled_hole_area_pixels += int(stats[label, cv2.CC_STAT_AREA])

    filled_mask = mask.astype(bool) | hole_mask
    return {
        "filled_mask": filled_mask.astype(bool),
        "hole_mask": hole_mask.astype(bool),
        "filled_hole_count": int(filled_hole_count),
        "filled_hole_area_pixels": int(filled_hole_area_pixels),
    }


def apply_local_contrast_normalization(gray_image: np.ndarray) -> np.ndarray:
    # 屋内外の照明差を少し吸収し、エッジを比較しやすくする。
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray_image)


def build_edge_persistence_map(
    frames: list[np.ndarray],
    max_samples: int = 36,
    canny_low_threshold: int = 60,
    canny_high_threshold: int = 160,
    persistence_threshold: float = EDGE_PERSISTENCE_THRESHOLD,
    endpoint_max_gap: int = EDGE_ENDPOINT_MAX_GAP,
    barrier_kernel_size: int = EDGE_BARRIER_KERNEL_SIZE,
    concave_hull_alpha_radius: float = CONCAVE_HULL_ALPHA_RADIUS,
    concave_hull_point_sample_step: int = CONCAVE_HULL_POINT_SAMPLE_STEP,
    min_edge_component_pixels: int = MIN_EDGE_COMPONENT_PIXELS,
    min_concave_hull_region_area_ratio: float = MIN_CONCAVE_HULL_REGION_AREA_RATIO,
) -> dict[str, Any]:
    sampled_frames, sampled_indices = sample_frames_evenly(frames, max_samples=max_samples)
    if not sampled_frames:
        return {
            "sampled_indices": [],
            "median_gray": None,
            "edge_persistence": None,
            "persistent_edge_mask": None,
            "endpoint_mask": None,
            "interpolated_edge_mask": None,
            "edge_barrier_mask": None,
            "concave_hull_candidate_mask": None,
            "concave_hull_hole_mask": None,
            "filled_concave_hull_mask": None,
            "vehicle_frame_region_mask": None,
            "persistence_threshold": float(persistence_threshold),
            "endpoint_max_gap": int(endpoint_max_gap),
            "barrier_kernel_size": int(barrier_kernel_size),
            "concave_hull_alpha_radius": float(concave_hull_alpha_radius),
            "concave_hull_point_sample_step": int(concave_hull_point_sample_step),
            "min_edge_component_pixels": int(min_edge_component_pixels),
            "min_concave_hull_region_area_ratio": float(min_concave_hull_region_area_ratio),
            "min_region_area_pixels": 0,
            "selected_region_count": 0,
            "edge_component_count": 0,
            "used_edge_component_count": 0,
            "inserted_point_count": 0,
            "kept_triangle_count": 0,
            "filled_hole_count": 0,
            "filled_hole_area_pixels": 0,
            "vehicle_frame_region_area_ratio": 0.0,
        }

    normalized_gray_frames: list[np.ndarray] = []
    edge_stack: list[np.ndarray] = []
    for frame in sampled_frames:
        gray_uint8 = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        blurred_gray = cv2.GaussianBlur(gray_uint8, (5, 5), 0)
        normalized_gray = apply_local_contrast_normalization(blurred_gray)
        normalized_gray_frames.append(normalized_gray.astype(np.float32))
        edge_stack.append(cv2.Canny(normalized_gray, canny_low_threshold, canny_high_threshold) > 0)

    gray_stack = np.stack(normalized_gray_frames, axis=0)
    median_gray = np.median(gray_stack, axis=0)
    edge_persistence = np.mean(np.stack(edge_stack, axis=0), axis=0)
    persistent_edge_mask = edge_persistence >= persistence_threshold
    interpolated_edge_mask, endpoint_mask = interpolate_edge_endpoints(
        persistent_edge_mask,
        max_gap=endpoint_max_gap,
    )
    edge_barrier_mask = build_edge_barrier_mask(
        interpolated_edge_mask,
        kernel_size=barrier_kernel_size,
    )
    concave_hull_info = build_concave_hull_candidate_mask(
        edge_barrier_mask,
        alpha_radius=concave_hull_alpha_radius,
        point_sample_step=concave_hull_point_sample_step,
        min_edge_component_pixels=min_edge_component_pixels,
    )
    hole_fill_info = fill_mask_holes(concave_hull_info["concave_hull_candidate_mask"])
    selected_region_info = select_large_concave_hull_regions(
        hole_fill_info["filled_mask"],
        min_area_ratio=min_concave_hull_region_area_ratio,
    )
    vehicle_frame_region_mask = selected_region_info["vehicle_frame_region_mask"]

    return {
        "sampled_indices": sampled_indices,
        "median_gray": median_gray.astype(np.float32),
        "edge_persistence": edge_persistence.astype(np.float32),
        "persistent_edge_mask": persistent_edge_mask.astype(bool),
        "endpoint_mask": endpoint_mask.astype(bool),
        "interpolated_edge_mask": interpolated_edge_mask.astype(bool),
        "edge_barrier_mask": edge_barrier_mask.astype(bool),
        "concave_hull_candidate_mask": concave_hull_info["concave_hull_candidate_mask"].astype(bool),
        "concave_hull_hole_mask": hole_fill_info["hole_mask"].astype(bool),
        "filled_concave_hull_mask": hole_fill_info["filled_mask"].astype(bool),
        "vehicle_frame_region_mask": vehicle_frame_region_mask.astype(bool),
        "persistence_threshold": float(persistence_threshold),
        "endpoint_max_gap": int(endpoint_max_gap),
        "barrier_kernel_size": int(barrier_kernel_size),
        "concave_hull_alpha_radius": float(concave_hull_alpha_radius),
        "concave_hull_point_sample_step": int(concave_hull_point_sample_step),
        "min_edge_component_pixels": int(min_edge_component_pixels),
        "min_concave_hull_region_area_ratio": float(min_concave_hull_region_area_ratio),
        "min_region_area_pixels": int(selected_region_info["min_region_area_pixels"]),
        "selected_region_count": int(selected_region_info["selected_region_count"]),
        "edge_component_count": int(concave_hull_info["edge_component_count"]),
        "used_edge_component_count": int(concave_hull_info["used_edge_component_count"]),
        "inserted_point_count": int(concave_hull_info["inserted_point_count"]),
        "kept_triangle_count": int(concave_hull_info["kept_triangle_count"]),
        "filled_hole_count": int(hole_fill_info["filled_hole_count"]),
        "filled_hole_area_pixels": int(hole_fill_info["filled_hole_area_pixels"]),
        "vehicle_frame_region_area_ratio": float(vehicle_frame_region_mask.mean()),
    }


In [ ]:
front_edge_debug_info = None
rear_edge_debug_info = None

if not reference_front_frames or not reference_rear_frames:
    print("Skip edge persistence preview because split frames are not available.")
else:
    front_edge_debug_info = build_edge_persistence_map(reference_front_frames)
    rear_edge_debug_info = build_edge_persistence_map(reference_rear_frames)

    print("edge persistence threshold:", front_edge_debug_info["persistence_threshold"])
    print("edge endpoint max gap:", front_edge_debug_info["endpoint_max_gap"])
    print("edge barrier kernel size:", front_edge_debug_info["barrier_kernel_size"])
    print("concave hull alpha radius:", front_edge_debug_info["concave_hull_alpha_radius"])
    print("concave hull point sample step:", front_edge_debug_info["concave_hull_point_sample_step"])
    print("min edge component pixels:", front_edge_debug_info["min_edge_component_pixels"])
    print("min concave hull region area ratio:", front_edge_debug_info["min_concave_hull_region_area_ratio"])
    print("front min region area pixels:", front_edge_debug_info["min_region_area_pixels"])
    print("rear min region area pixels:", rear_edge_debug_info["min_region_area_pixels"])
    print("front edge component count:", front_edge_debug_info["edge_component_count"])
    print("rear edge component count:", rear_edge_debug_info["edge_component_count"])
    print("front used edge component count:", front_edge_debug_info["used_edge_component_count"])
    print("rear used edge component count:", rear_edge_debug_info["used_edge_component_count"])
    print("front inserted point count:", front_edge_debug_info["inserted_point_count"])
    print("rear inserted point count:", rear_edge_debug_info["inserted_point_count"])
    print("front kept triangle count:", front_edge_debug_info["kept_triangle_count"])
    print("rear kept triangle count:", rear_edge_debug_info["kept_triangle_count"])
    print("front filled hole count:", front_edge_debug_info["filled_hole_count"])
    print("rear filled hole count:", rear_edge_debug_info["filled_hole_count"])
    print("front filled hole area pixels:", front_edge_debug_info["filled_hole_area_pixels"])
    print("rear filled hole area pixels:", rear_edge_debug_info["filled_hole_area_pixels"])
    print("front selected region count:", front_edge_debug_info["selected_region_count"])
    print("rear selected region count:", rear_edge_debug_info["selected_region_count"])
    print("front region area ratio:", front_edge_debug_info["vehicle_frame_region_area_ratio"])
    print("rear region area ratio:", rear_edge_debug_info["vehicle_frame_region_area_ratio"])
    print("front sampled indices:", front_edge_debug_info["sampled_indices"])
    print("rear sampled indices:", rear_edge_debug_info["sampled_indices"])

    show_image(normalize_map_to_rgb(front_edge_debug_info["median_gray"]), title="Front Median Gray")
    show_image(normalize_map_to_rgb(front_edge_debug_info["edge_persistence"]), title="Front Edge Persistence")
    show_image(normalize_map_to_rgb(front_edge_debug_info["persistent_edge_mask"].astype(np.float32)), title="Front Binarized Edge Persistence")
    show_image(normalize_map_to_rgb(front_edge_debug_info["endpoint_mask"].astype(np.float32)), title="Front Edge Endpoints")
    show_image(normalize_map_to_rgb(front_edge_debug_info["interpolated_edge_mask"].astype(np.float32)), title="Front Endpoint-Interpolated Edge Mask")
    show_image(normalize_map_to_rgb(front_edge_debug_info["edge_barrier_mask"].astype(np.float32)), title="Front Edge Barrier Mask")
    show_image(normalize_map_to_rgb(front_edge_debug_info["concave_hull_candidate_mask"].astype(np.float32)), title="Front Concave Hull Candidates")
    show_image(normalize_map_to_rgb(front_edge_debug_info["concave_hull_hole_mask"].astype(np.float32)), title="Front Concave Hull Hole Mask")
    show_image(normalize_map_to_rgb(front_edge_debug_info["filled_concave_hull_mask"].astype(np.float32)), title="Front Hole-Filled Concave Hull")
    show_image(normalize_map_to_rgb(front_edge_debug_info["vehicle_frame_region_mask"].astype(np.float32)), title="Front Selected Vehicle Frame Region")
    show_image(
        draw_mask_overlay(reference_front_frame, front_edge_debug_info["interpolated_edge_mask"]),
        title="Front Endpoint-Interpolated Edge Mask Overlay",
    )
    show_image(
        draw_mask_overlay(reference_front_frame, front_edge_debug_info["vehicle_frame_region_mask"]),
        title="Front Vehicle Frame Region Overlay",
    )

    show_image(normalize_map_to_rgb(rear_edge_debug_info["median_gray"]), title="Rear Median Gray")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["edge_persistence"]), title="Rear Edge Persistence")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["persistent_edge_mask"].astype(np.float32)), title="Rear Binarized Edge Persistence")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["endpoint_mask"].astype(np.float32)), title="Rear Edge Endpoints")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["interpolated_edge_mask"].astype(np.float32)), title="Rear Endpoint-Interpolated Edge Mask")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["edge_barrier_mask"].astype(np.float32)), title="Rear Edge Barrier Mask")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["concave_hull_candidate_mask"].astype(np.float32)), title="Rear Concave Hull Candidates")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["concave_hull_hole_mask"].astype(np.float32)), title="Rear Concave Hull Hole Mask")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["filled_concave_hull_mask"].astype(np.float32)), title="Rear Hole-Filled Concave Hull")
    show_image(normalize_map_to_rgb(rear_edge_debug_info["vehicle_frame_region_mask"].astype(np.float32)), title="Rear Selected Vehicle Frame Region")
    show_image(
        draw_mask_overlay(reference_rear_frame, rear_edge_debug_info["interpolated_edge_mask"]),
        title="Rear Endpoint-Interpolated Edge Mask Overlay",
    )
    show_image(
        draw_mask_overlay(reference_rear_frame, rear_edge_debug_info["vehicle_frame_region_mask"]),
        title="Rear Vehicle Frame Region Overlay",
    )
